# Guitar Tab Transcription - Clean Demo

Single file inference with custom AR generation and visualization

In [1]:
import sys
from pathlib import Path
import json
import random
import torch
import numpy as np

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))

from src.dadagp_parser import parse_dadagp_file, dadagp_to_events
from src.tab_dataset import (
    build_vocabulary, events_to_ids, event_to_token_string,
    NoteOnEvent, NoteOffEvent, TimeShiftEvent, TabEvent
)
from src.model import FrettingTransformer
from src.metrics import compute_tablature_accuracy
from src.visualization import render_as_tablature, render_as_notes

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

/tmp2/b10401006/.symlinks/miniforge3/envs/guitar-tab/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


In [4]:

import os 
if Path.cwd().name == "notebooks":
    os.chdir(Path(os.getcwd()).parent)

## Configuration

In [5]:
# Paths
DATA_DIR = Path("./DadaGP")
TEST_FILES_JSON = Path("./data_splits/test_files.json")
CHECKPOINT_PATH = Path("./outputs/2025-12-05_03-25/best_model.pt")  # UPDATE THIS

# Model config (should match training)
MODEL_CONFIG = {
    'd_model': 128,
    'd_ff': 1024,
    'num_layers': 3,
    'num_heads': 4,
    'dropout_rate': 0.1
}

# Generation config
MAX_LENGTH = 1024
NUM_BEAMS = 1
TEMPERATURE = 1.0

## Load and Process Data

In [6]:
# Load test files list
with open(TEST_FILES_JSON, 'r') as f:
    test_files = json.load(f)

# Pick a random test file
random.seed(42)
selected_file = random.choice(test_files)
token_file = selected_file + ".tokens.txt"

print(f"Selected file: {Path(selected_file).name}")

# Parse DadaGP tokens
dadagp_tokens = parse_dadagp_file(token_file)
input_events, output_events, bar_positions = dadagp_to_events(dadagp_tokens)

print(f"Input events: {len(input_events)}")
print(f"Output events: {len(output_events)}")

# Build vocabularies
input_vocab, output_vocab = build_vocabulary(
    max_pitch=127,
    max_time_shift=500,
    num_strings=6,
    num_frets=21
)

# Convert to token IDs
input_ids = events_to_ids(input_events, input_vocab)
output_ids = events_to_ids(output_events, output_vocab)

# Truncate if needed
if len(input_ids) > MAX_LENGTH:
    input_ids = input_ids[:MAX_LENGTH]
    input_events = input_events[:MAX_LENGTH]

if len(output_ids) > MAX_LENGTH:
    output_ids = output_ids[:MAX_LENGTH]
    output_events = output_events[:MAX_LENGTH]

# Convert to tensors
input_tensor = torch.tensor(input_ids, dtype=torch.long).unsqueeze(0).to(device)
target_tensor = torch.tensor(output_ids, dtype=torch.long).unsqueeze(0).to(device)

print(f"\nInput shape: {input_tensor.shape}")
print(f"Target shape: {target_tensor.shape}")

Selected file: Yoshimatsu, Takashi - Angels In Twilight.gp4
Input events: 673
Output events: 927

Input shape: torch.Size([1, 673])
Target shape: torch.Size([1, 927])


## Load Model

In [7]:
# Create model
model = FrettingTransformer(
    input_vocab_size=input_vocab.vocab_size,
    output_vocab_size=output_vocab.vocab_size,
    model_config=MODEL_CONFIG
).to(device)

# Load checkpoint
if CHECKPOINT_PATH.exists():
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"Loaded checkpoint from epoch {checkpoint['epoch']}")
    print(f"Val loss: {checkpoint['val_loss']:.4f}")
else:
    raise FileNotFoundError(f"Checkpoint not found: {CHECKPOINT_PATH}")

model.eval()
print("Model ready")

Created custom T5 model:
  Encoder vocab: 760
  Decoder vocab: 886
  d_model: 128
  d_ff: 1024
  layers: 3
  heads: 4
  parameters: 2,489,216
Loaded checkpoint from epoch 20
Val loss: 0.0320
Model ready


## Custom AR Generation Function

In [8]:
def custom_generate(
    model,
    input_ids,
    attention_mask=None,
    max_length=100,
    start_token_id=None,
    eos_token_id=2,
    temperature=1.0,
    verbose=False
):
    """
    Custom autoregressive generation with proper attention handling.
    
    Args:
        model: FrettingTransformer model
        input_ids: [B, L_enc] - Encoder input
        attention_mask: [B, L_enc] - Encoder attention mask
        max_length: Maximum generation length
        start_token_id: First decoder token (use first target token for old checkpoints)
        eos_token_id: End-of-sequence token ID
        temperature: Sampling temperature
        verbose: Print generation progress
    
    Returns:
        [B, L_gen] - Generated token IDs (includes start token)
    """
    device = input_ids.device
    batch_size = input_ids.shape[0]
    
    if verbose:
        print(f"Starting AR generation...")
        print(f"  Start token: {start_token_id}")
        print(f"  Max length: {max_length}")
    
    # Encode input once
    with torch.no_grad():
        encoder_outputs = model.model.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
    
    # Initialize decoder with start token
    decoder_input_ids = torch.full(
        (batch_size, 1),
        start_token_id,
        dtype=torch.long,
        device=device
    )
    
    generated_tokens = []
    
    # Autoregressive generation
    for step in range(max_length):
        with torch.no_grad():
            # Create decoder attention mask (all 1s for generated tokens)
            decoder_attention_mask = torch.ones_like(decoder_input_ids)
            
            # Forward pass
            outputs = model.model(
                encoder_outputs=encoder_outputs,
                decoder_input_ids=decoder_input_ids,
                decoder_attention_mask=decoder_attention_mask,
                attention_mask=attention_mask,
            )
            
            # Get logits for last position
            logits = outputs.logits[:, -1, :]
            
            # Apply temperature
            if temperature != 1.0:
                logits = logits / temperature
            
            # Greedy decoding
            probs = torch.softmax(logits, dim=-1)
            next_token = torch.argmax(probs, dim=-1)
            
            # Append to sequence
            generated_tokens.append(next_token)
            
            # Check for EOS
            if (next_token == eos_token_id).all():
                if verbose:
                    print(f"  EOS at step {step}")
                break
            
            # Append to decoder input for next iteration
            decoder_input_ids = torch.cat([
                decoder_input_ids,
                next_token.unsqueeze(1)
            ], dim=1)
    
    # Concatenate all generated tokens
    generated_sequence = torch.stack(generated_tokens, dim=1)
    
    # Prepend start token
    full_sequence = torch.cat([
        torch.full((batch_size, 1), start_token_id, dtype=torch.long, device=device),
        generated_sequence
    ], dim=1)
    
    if verbose:
        print(f"  Generated {len(generated_tokens)} tokens")
    
    return full_sequence

## Run Generation (Teacher Forcing First Token)

In [9]:
# Get first target token for teacher forcing
first_target_token = target_tensor[0, 0].item()
print(f"Using first target token as start: {first_target_token} ({output_vocab.id_to_token.get(first_target_token, 'UNK')})")
print()

# Generate
prediction = custom_generate(
    model=model,
    input_ids=input_tensor,
    max_length=target_tensor.shape[1],
    start_token_id=first_target_token,
    eos_token_id=output_vocab.eos_id,
    temperature=TEMPERATURE,
    verbose=True
)

print(f"\nGenerated shape: {prediction.shape}")
print(f"Target shape: {target_tensor.shape}")

Using first target token as start: 44 (NOTE_ON_40)

Starting AR generation...
  Start token: 44
  Max length: 927
  Generated 927 tokens

Generated shape: torch.Size([1, 928])
Target shape: torch.Size([1, 927])


## Compute Metrics

In [10]:
# Pad/trim prediction to match target length
target_len = target_tensor.shape[1]
pred_len = prediction.shape[1]

if pred_len < target_len:
    padding = torch.full(
        (1, target_len - pred_len),
        output_vocab.pad_id,
        dtype=prediction.dtype,
        device=prediction.device
    )
    prediction_padded = torch.cat([prediction, padding], dim=1)
elif pred_len > target_len:
    prediction_padded = prediction[:, :target_len]
else:
    prediction_padded = prediction

# Compute accuracy
metrics = compute_tablature_accuracy(
    predictions=prediction_padded,
    targets=target_tensor,
    output_vocab=output_vocab,
    pad_id=output_vocab.pad_id
)

print("="*60)
print("METRICS")
print("="*60)
print(f"Token Accuracy:  {metrics.token_accuracy:.2%}")
print(f"Pitch Accuracy:  {metrics.pitch_accuracy:.2%}")
print(f"Tab Accuracy:    {metrics.tab_accuracy:.2%}")
print(f"Total Tokens:    {metrics.total_tokens:,}")
print(f"Total Notes:     {metrics.total_notes:,}")
print("="*60)

METRICS
Token Accuracy:  19.09%
Pitch Accuracy:  18.50%
Tab Accuracy:    14.96%
Total Tokens:    927
Total Notes:     254


## Visualization

In [11]:
def ids_to_events(token_ids, vocab):
    """Convert token IDs back to Event objects."""
    events = []
    
    for token_id in token_ids:
        if token_id == vocab.pad_id:
            break
        
        token_str = vocab.id_to_token.get(token_id, "UNK")
        
        if token_str.startswith("NOTE_ON_"):
            pitch = int(token_str.split("_")[-1])
            events.append(NoteOnEvent(pitch=pitch))
        elif token_str.startswith("NOTE_OFF_"):
            pitch = int(token_str.split("_")[-1])
            events.append(NoteOffEvent(pitch=pitch))
        elif token_str.startswith("TIME_SHIFT_"):
            delta = int(token_str.split("_")[-1])
            events.append(TimeShiftEvent(delta=delta))
        elif token_str.startswith("TAB_"):
            parts = token_str.split("_")
            string = int(parts[1])
            fret = int(parts[2])
            events.append(TabEvent(string=string, fret=fret))
    
    return events

# Convert predictions to events
pred_events = ids_to_events(prediction[0].cpu().tolist(), output_vocab)
print(f"Converted {len(pred_events)} prediction events")

Converted 928 prediction events


### Input (MIDI Notes Only)

In [24]:
print(render_as_notes(input_events, max_bars=12, chars_per_beat=8, bars_per_row=4))

Note Notation (like tablature, showing note names)
 |A3--A2--C3------A2------D4--G#3-|D3------E3------D3--A2--C3------|G2------B3--A2--D3------C3------|A3--A2--C3------A2------D4--G#3-|
 |             Bar 1              |             Bar 2              |             Bar 3              |             Bar 4              |

 |D3------E3------D3--A2--A43-----|E3------B3--A2--A3--------------|G4--A2--A3------B2------A3--E3--|B2------E2------A3--E3--A#2-----|
 |             Bar 5              |             Bar 6              |             Bar 7              |             Bar 8              |

 |E2------D4--D3--A2------C#4-----|A3--D3--A2------A2------D4--G#3-|D3------A3------C4--F#3-C3------|B3--G2--A3--A2--D3--------------|
 |             Bar 9              |             Bar 10             |             Bar 11             |             Bar 12             |

... (18 more bars not shown)


### Targets vs. predictions

In [23]:
print(render_as_notes(output_events, max_bars=12, chars_per_beat=8, bars_per_row=4))
print(render_as_notes(pred_events, max_bars=12, chars_per_beat=8, bars_per_row=4))

Note Notation (like tablature, showing note names)
 |A3--A2--C3------A2------D4--G#3-|D3------E3------D3--A2--C3------|G2------B3--A2--D3------C3------|A3--A2--C3------A2------D4--G#3-|
 |             Bar 1              |             Bar 2              |             Bar 3              |             Bar 4              |

 |D3------E3------D3--A2--A43-----|E3------B3--A2--A3--------------|G4--A2--A3------B2------A3--E3--|B2------E2------A3--E3--A#2-----|
 |             Bar 5              |             Bar 6              |             Bar 7              |             Bar 8              |

 |E2------D4--D3--A2------C#4-----|A3--D3--A2------A2------D4--G#3-|D3------A3------C4--F#3-C3------|B3--G2--A3--A2--D3--------------|
 |             Bar 9              |             Bar 10             |             Bar 11             |             Bar 12             |

... (18 more bars not shown)
Note Notation (like tablature, showing note names)
 |A3--A2--C3------A2------D4--G#3-|D3------E3------D3--A

In [22]:
print(render_as_tablature(output_events, max_bars=12, chars_per_beat=8, bars_per_row=4))
print(render_as_tablature(pred_events, max_bars=12, chars_per_beat=8, bars_per_row=4))

Guitar Tablature (Standard Tuning: EADGBE)
e|0-------3-------5-------7-------|--------12------7-------0-------|3-------0-----------------------|0-------3-------5-------7-------|
B|----0---3-----------------------|5-------------------0---3-------|------------0---5-------3-------|----0---3-----------------------|
G|----------------------------6---|----------------0---------------|--------------------------------|----------------------------6---|
D|2-----------------------7-------|--------------------------------|--------4-----------------------|2-----------------------7-------|
A|--------------------------------|--------------------------------|--------------------------------|--------------------------------|
E|--------------------------------|--------------------------------|--------------------------------|--------------------------------|
 |             Bar 1              |             Bar 2              |             Bar 3              |             Bar 4              |

e|--------1

## Token-by-Token Comparison (First 30)

In [17]:
print("="*90)
print("FIRST 30 TOKENS COMPARISON")
print("="*90)
print(f"{'#':<4} {'TARGET':<35} {'PREDICTION':<35} {'MATCH'}")
print("-"*90)

for i in range(min(30, target_tensor.shape[1])):
    target_id = target_tensor[0, i].item()
    pred_id = prediction_padded[0, i].item() if i < prediction_padded.shape[1] else output_vocab.pad_id
    
    target_str = output_vocab.id_to_token.get(target_id, "UNK")
    pred_str = output_vocab.id_to_token.get(pred_id, "UNK")
    
    match = "✓" if target_id == pred_id else "✗"
    print(f"{i:<4} {target_str:<35} {pred_str:<35} {match}")

print("="*90)

FIRST 30 TOKENS COMPARISON
#    TARGET                              PREDICTION                          MATCH
------------------------------------------------------------------------------------------
0    NOTE_ON_40                          NOTE_ON_40                          ✓
1    TAB_1_0                             TAB_1_0                             ✓
2    NOTE_ON_57                          NOTE_ON_57                          ✓
3    TAB_4_2                             TAB_4_2                             ✓
4    NOTE_OFF_40                         NOTE_OFF_40                         ✓
5    NOTE_OFF_57                         NOTE_OFF_57                         ✓
6    TIME_SHIFT_240                      TIME_SHIFT_240                      ✓
7    NOTE_ON_45                          NOTE_ON_45                          ✓
8    TAB_2_0                             TAB_2_0                             ✓
9    NOTE_OFF_45                         NOTE_OFF_45                         ✓
10   TIME